# DeepTrace — Module 1: URL Phishing Training
## XGBoost + LightGBM Ensemble | CPU Runtime | ~20 min

**Before running:** Upload `url_processed.zip` from notebook 01 when Cell 2 asks.

In [ ]:
# CELL 1 — Install
!pip install -q xgboost lightgbm optuna scikit-learn joblib matplotlib

In [ ]:
# CELL 2 — Upload + extract processed data
from google.colab import files
import zipfile, os

print('Upload url_processed.zip')
uploaded = files.upload()
os.makedirs('/content/data/processed', exist_ok=True)
with zipfile.ZipFile('url_processed.zip','r') as z:
    z.extractall('/content/data/processed/')
print('Extracted:')
!ls -lh /content/data/processed/

In [ ]:
# CELL 3 — Load arrays
import numpy as np, json
from collections import Counter

base = '/content/data/processed'
X_tr = np.load(f'{base}/X_train.npy')
X_va = np.load(f'{base}/X_val.npy')
X_te = np.load(f'{base}/X_test.npy')
y_tr = np.load(f'{base}/y_train.npy').astype(int)
y_va = np.load(f'{base}/y_val.npy').astype(int)
y_te = np.load(f'{base}/y_test.npy').astype(int)

with open(f'{base}/feature_names.json') as f:
    FEAT = json.load(f)

# Clean any NaN/Inf
for arr in [X_tr, X_va, X_te]:
    np.nan_to_num(arr, copy=False, nan=0.0, posinf=0.0, neginf=0.0)

X_all = np.vstack([X_tr, X_va])
y_all = np.concatenate([y_tr, y_va])

print(f'Train: {X_tr.shape}  Val: {X_va.shape}  Test: {X_te.shape}')
print(f'Train labels: {Counter(y_tr)}')
print(f'Feature count: {len(FEAT)}')
assert X_tr.shape[1] == len(FEAT) == 52

In [ ]:
# CELL 4 — XGBoost Optuna (30 trials)
# Tree models do NOT need StandardScaler — using raw features
import time, optuna, xgboost as xgb
from sklearn.metrics import roc_auc_score
optuna.logging.set_verbosity(optuna.logging.WARNING)

def xgb_obj(trial):
    m = xgb.XGBClassifier(
        n_estimators     = trial.suggest_int('n_est', 200, 500),
        max_depth        = trial.suggest_int('depth', 4, 9),
        learning_rate    = trial.suggest_float('lr', 0.02, 0.25, log=True),
        subsample        = trial.suggest_float('sub', 0.65, 1.0),
        colsample_bytree = trial.suggest_float('col', 0.65, 1.0),
        min_child_weight = trial.suggest_int('mcw', 1, 10),
        gamma            = trial.suggest_float('gam', 0.0, 0.4),
        reg_alpha        = trial.suggest_float('a', 0.0, 0.8),
        reg_lambda       = trial.suggest_float('l', 0.8, 2.0),
        eval_metric='logloss', random_state=42,
        tree_method='hist', n_jobs=-1
    )
    m.fit(X_tr, y_tr, eval_set=[(X_va,y_va)], verbose=False)
    return roc_auc_score(y_va, m.predict_proba(X_va)[:,1])

print('XGBoost Optuna — 30 trials...')
t0 = time.time()
xgb_study = optuna.create_study(direction='maximize')
xgb_study.optimize(xgb_obj, n_trials=30, show_progress_bar=True)
print(f'Done in {(time.time()-t0)/60:.1f} min')
print(f'XGB best AUC: {xgb_study.best_value:.4f}')
print(xgb_study.best_params)

In [ ]:
# CELL 5 — LightGBM Optuna (30 trials)
import lightgbm as lgb

def lgb_obj(trial):
    m = lgb.LGBMClassifier(
        n_estimators     = trial.suggest_int('n_est', 200, 500),
        max_depth        = trial.suggest_int('depth', 4, 9),
        learning_rate    = trial.suggest_float('lr', 0.02, 0.25, log=True),
        subsample        = trial.suggest_float('sub', 0.65, 1.0),
        colsample_bytree = trial.suggest_float('col', 0.65, 1.0),
        min_child_samples= trial.suggest_int('mcs', 5, 40),
        num_leaves       = trial.suggest_int('leaves', 25, 120),
        reg_alpha        = trial.suggest_float('a', 0.0, 0.8),
        reg_lambda       = trial.suggest_float('l', 0.8, 2.0),
        random_state=42, n_jobs=-1, verbose=-1
    )
    m.fit(X_tr, y_tr,
          eval_set=[(X_va,y_va)],
          callbacks=[lgb.early_stopping(25,verbose=False), lgb.log_evaluation(-1)])
    return roc_auc_score(y_va, m.predict_proba(X_va)[:,1])

print('LightGBM Optuna — 30 trials...')
t0 = time.time()
lgb_study = optuna.create_study(direction='maximize')
lgb_study.optimize(lgb_obj, n_trials=30, show_progress_bar=True)
print(f'Done in {(time.time()-t0)/60:.1f} min')
print(f'LGB best AUC: {lgb_study.best_value:.4f}')
print(lgb_study.best_params)

In [ ]:
# CELL 6 — Train final models on train+val combined
xp = xgb_study.best_params
XGB = xgb.XGBClassifier(
    n_estimators=xp['n_est'], max_depth=xp['depth'],
    learning_rate=xp['lr'], subsample=xp['sub'],
    colsample_bytree=xp['col'], min_child_weight=xp['mcw'],
    gamma=xp['gam'], reg_alpha=xp['a'], reg_lambda=xp['l'],
    eval_metric='logloss', random_state=42, tree_method='hist', n_jobs=-1
)
XGB.fit(X_all, y_all)
print('✓ XGBoost trained on full train+val')

lp = lgb_study.best_params
LGB = lgb.LGBMClassifier(
    n_estimators=lp['n_est'], max_depth=lp['depth'],
    learning_rate=lp['lr'], subsample=lp['sub'],
    colsample_bytree=lp['col'], min_child_samples=lp['mcs'],
    num_leaves=lp['leaves'], reg_alpha=lp['a'], reg_lambda=lp['l'],
    random_state=42, n_jobs=-1, verbose=-1
)
LGB.fit(X_all, y_all)
print('✓ LightGBM trained on full train+val')

In [ ]:
# CELL 7 — Best threshold on validation, then evaluate test
import numpy as np
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score, classification_report, confusion_matrix

# Retrain on train only to get val predictions for threshold selection
XGB_v = xgb.XGBClassifier(
    n_estimators=xp['n_est'], max_depth=xp['depth'], learning_rate=xp['lr'],
    subsample=xp['sub'], colsample_bytree=xp['col'], min_child_weight=xp['mcw'],
    gamma=xp['gam'], reg_alpha=xp['a'], reg_lambda=xp['l'],
    eval_metric='logloss', random_state=42, tree_method='hist', n_jobs=-1
)
XGB_v.fit(X_tr, y_tr)

LGB_v = lgb.LGBMClassifier(
    n_estimators=lp['n_est'], max_depth=lp['depth'], learning_rate=lp['lr'],
    subsample=lp['sub'], colsample_bytree=lp['col'], min_child_samples=lp['mcs'],
    num_leaves=lp['leaves'], reg_alpha=lp['a'], reg_lambda=lp['l'],
    random_state=42, n_jobs=-1, verbose=-1
)
LGB_v.fit(X_tr, y_tr)

val_p = 0.5*XGB_v.predict_proba(X_va)[:,1] + 0.5*LGB_v.predict_proba(X_va)[:,1]

best_thr, best_f1 = 0.5, 0.0
for t in np.arange(0.20, 0.81, 0.01):
    f = f1_score(y_va, (val_p>=t).astype(int))
    if f > best_f1: best_f1, best_thr = f, float(t)

print(f'Best val threshold: {best_thr:.3f}  (F1={best_f1:.4f})')

# Test with full train+val models
te_p = 0.5*XGB.predict_proba(X_te)[:,1] + 0.5*LGB.predict_proba(X_te)[:,1]
yp   = (te_p>=best_thr).astype(int)

acc = accuracy_score(y_te, yp)
f1  = f1_score(y_te, yp)
auc = roc_auc_score(y_te, te_p)

print('='*55)
print(f'TEST  Accuracy : {acc:.4f}')
print(f'TEST  F1       : {f1:.4f}')
print(f'TEST  ROC-AUC  : {auc:.4f}')
print('='*55)
print(classification_report(y_te, yp, target_names=['Legitimate','Phishing']))
print('Confusion matrix:\n', confusion_matrix(y_te, yp))

In [ ]:
# CELL 8 — Feature importance plot
import matplotlib.pyplot as plt, numpy as np, os
os.makedirs('/content/url_models', exist_ok=True)

xi = XGB.feature_importances_ / (XGB.feature_importances_.sum()+1e-9)
li = LGB.feature_importances_.astype(float); li /= (li.sum()+1e-9)
avg = (xi+li)/2
idx = np.argsort(avg)[-20:]

plt.figure(figsize=(10,7))
plt.barh(np.array(FEAT)[idx], avg[idx])
plt.xlabel('Average normalized importance')
plt.title('Top 20 URL Features (XGB + LGB ensemble)')
plt.tight_layout()
plt.savefig('/content/url_models/feature_importance.png', dpi=150)
plt.show()
print('✓ Plot saved')

In [ ]:
# CELL 9 — Save models + download
import joblib, json, shutil, os
from google.colab import files

out = '/content/url_models'
os.makedirs(out, exist_ok=True)

joblib.dump(XGB,  f'{out}/xgb_url.pkl')
joblib.dump(LGB,  f'{out}/lgb_url.pkl')
joblib.dump(None, f'{out}/url_scaler.pkl')   # tree models need no scaler

with open(f'{out}/feature_names.json','w') as f:
    json.dump(FEAT, f, indent=2)

cfg = {
    'threshold': float(best_thr),
    'xgb_weight': 0.5, 'lgb_weight': 0.5,
    'feature_count': len(FEAT),
    'test_auc': float(auc),
    'test_f1':  float(f1),
    'test_acc': float(acc)
}
with open(f'{out}/url_model_config.json','w') as f:
    json.dump(cfg, f, indent=2)

print('Config:', json.dumps(cfg, indent=2))

shutil.make_archive('/content/url_models_pkg','zip', out)
files.download('/content/url_models_pkg.zip')
print('\n✅ Extract into: DeepTrace/training/module1_url/models/')